# Data Profiling
## Indonesia Palm Oil Supply Chain Dataset

Notebook ini digunakan untuk melakukan profiling awal terhadap dataset
**Indonesia Palm Oil Supply Chain** dari Trase.

Profiling dilakukan sebagai bagian dari **Milestone 1 Tugas 1 Analisis Big Data 2026**.

### Tujuan

1. Memastikan dataset memenuhi ketentuan ukuran tugas.
2. Mengetahui struktur dan karakteristik dataset.
3. Mengidentifikasi nama kolom dan tipe data.
4. Memeriksa missing values secara awal.
5. Memeriksa data duplikat.
6. Mengetahui jumlah nilai unik pada setiap kolom.
7. Melakukan profiling sederhana menggunakan Polars.
8. Melakukan analisis SQL awal menggunakan DuckDB.

### Informasi Dataset

- **Nama:** Indonesia Palm Oil - Supply Chain Data
- **Sumber:** Trase
- **Periode:** 2013–2022
- **File:** `indonesia_palm_oil_v1_2_5.csv`
- **Format:** CSV
- **Unit analisis:** Data rantai pasok minyak kelapa sawit Indonesia

### Sumber Dataset

https://trase.earth/open-data/datasets/supply-chains-indonesia-palm-oil

In [ ]:
import polars as pl
import duckdb
import os
import time

print("Polars version :", pl.__version__)
print("DuckDB version :", duckdb.__version__)

In [ ]:
DATA_PATH = r"C:\Users\AYESHA NINGRUM\tugas1-abd-2026\data\raw\indonesia_palm_oil_v1_2_5.csv"

print("Dataset:")
print(DATA_PATH)

print("\nFile tersedia:", os.path.exists(DATA_PATH))

if os.path.exists(DATA_PATH):
    file_size = os.path.getsize(DATA_PATH)

    print(
        "Ukuran file:",
        f"{file_size / (1024**2):,.2f} MB"
    )
else:
    print("ERROR: File dataset tidak ditemukan.")

In [ ]:
file_size = os.path.getsize(DATA_PATH)

file_size_mb = file_size / (1024 ** 2)
file_size_gb = file_size / (1024 ** 3)

print(f"Ukuran dataset : {file_size_mb:,.2f} MB")
print(f"Ukuran dataset : {file_size_gb:,.2f} GB")

In [ ]:
start_time = time.time()

lf = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10_000,
    ignore_errors=False
)

elapsed = time.time() - start_time

print("LazyFrame berhasil dibuat.")
print(f"Waktu inisialisasi: {elapsed:.2f} detik")

In [ ]:
schema = lf.collect_schema()

schema

In [ ]:
schema_df = pl.DataFrame({
    "column": list(schema.names()),
    "dtype": [
        str(schema[name])
        for name in schema.names()
    ]
})

schema_df

In [ ]:
column_count = len(schema.names())

print("Jumlah kolom:", column_count)

In [ ]:
start_time = time.time()

row_count = (
    lf.select(
        pl.len().alias("jumlah_baris")
    )
    .collect()
    .item()
)

elapsed = time.time() - start_time

print("Jumlah baris:", f"{row_count:,}")
print(f"Waktu proses: {elapsed:.2f} detik")

In [ ]:
print("HASIL VALIDASI DATASET")
print("=" * 40)

print(f"Jumlah baris : {row_count:,}")
print(f"Ukuran file : {file_size_mb:,.2f} MB")

if row_count > 1_000_000:
    print("\n✓ MEMENUHI: jumlah baris > 1.000.000")
elif file_size_mb >= 500:
    print("\n✓ MEMENUHI: ukuran file >= 500 MB")
else:
    print("\n✗ BELUM MEMENUHI ketentuan ukuran dataset")

In [ ]:
sample = (
    lf
    .head(10)
    .collect()
)

sample

In [ ]:
schema_df

In [ ]:
missing_profile = (
    lf
    .select([
        pl.col(column)
        .null_count()
        .alias(column)
        for column in schema.names()
    ])
    .collect()
)

missing_profile

In [ ]:
missing_summary = pl.DataFrame({
    "column": schema.names(),
    "missing_count": [
        missing_profile[column][0]
        for column in schema.names()
    ]
})

missing_summary = missing_summary.with_columns(
    (
        pl.col("missing_count") / row_count * 100
    ).alias("missing_percentage")
)

missing_summary.sort(
    "missing_count",
    descending=True
)

In [ ]:
missing_columns = (
    missing_summary
    .filter(
        pl.col("missing_count") > 0
    )
    .sort(
        "missing_count",
        descending=True
    )
)

print(
    "Jumlah kolom dengan missing value:",
    missing_columns.height
)

missing_columns

In [ ]:
start_time = time.time()

unique_row_count = (
    lf
    .unique()
    .select(
        pl.len().alias("unique_rows")
    )
    .collect()
    .item()
)

duplicate_count = row_count - unique_row_count

elapsed = time.time() - start_time

print("Jumlah baris       :", f"{row_count:,}")
print("Jumlah baris unik  :", f"{unique_row_count:,}")
print("Perkiraan duplikat :", f"{duplicate_count:,}")
print(f"Waktu proses       : {elapsed:.2f} detik")

In [ ]:
unique_counts = (
    lf
    .select([
        pl.col(column)
        .n_unique()
        .alias(column)
        for column in schema.names()
    ])
    .collect()
)

unique_summary = pl.DataFrame({
    "column": schema.names(),
    "unique_values": [
        unique_counts[column][0]
        for column in schema.names()
    ]
})

unique_summary.sort(
    "unique_values",
    descending=True
)

In [ ]:
numeric_columns = [
    name
    for name, dtype in schema.items()
    if dtype.is_numeric()
]

print("Kolom numerik:")
for column in numeric_columns:
    print("-", column)

print("\nJumlah kolom numerik:", len(numeric_columns))

In [ ]:
categorical_columns = [
    name
    for name, dtype in schema.items()
    if dtype == pl.String
]

print("Kolom kategorikal/string:")

for column in categorical_columns:
    print("-", column)

print(
    "\nJumlah kolom kategorikal:",
    len(categorical_columns)
)

In [ ]:
if numeric_columns:

    numeric_summary = (
        lf
        .select(numeric_columns)
        .collect()
        .describe()
    )

    numeric_summary

else:
    print("Tidak terdapat kolom numerik.")

In [ ]:
if "year" in schema.names():

    year_profile = (
        lf
        .group_by("year")
        .agg(
            pl.len().alias("jumlah_baris")
        )
        .sort("year")
        .collect()
    )

    year_profile

else:
    print("Kolom 'year' tidak ditemukan.")

In [ ]:
if "year" in schema.names():

    year_range = (
        lf
        .select([
            pl.col("year").min().alias("tahun_minimum"),
            pl.col("year").max().alias("tahun_maksimum")
        ])
        .collect()
    )

    year_range

In [ ]:
if "country_of_production" in schema.names():

    country_profile = (
        lf
        .group_by("country_of_production")
        .agg(
            pl.len().alias("jumlah_baris")
        )
        .sort(
            "jumlah_baris",
            descending=True
        )
        .head(20)
        .collect()
    )

    country_profile

else:
    print(
        "Kolom 'country_of_production' tidak ditemukan."
    )

In [ ]:
if "product_type" in schema.names():

    product_profile = (
        lf
        .group_by("product_type")
        .agg(
            pl.len().alias("jumlah_baris")
        )
        .sort(
            "jumlah_baris",
            descending=True
        )
        .head(20)
        .collect()
    )

    product_profile

else:
    print(
        "Kolom 'product_type' tidak ditemukan."
    )

In [ ]:
con = duckdb.connect()

print("DuckDB berhasil terhubung.")

In [ ]:
duckdb_result = con.execute(f"""
    SELECT *
    FROM read_csv_auto('{DATA_PATH}')
    LIMIT 10
""").df()

duckdb_result

In [ ]:
duckdb_count = con.execute(f"""
    SELECT COUNT(*) AS jumlah_baris
    FROM read_csv_auto('{DATA_PATH}')
""").df()

duckdb_count

In [ ]:
summarize_result = con.execute(f"""
    SUMMARIZE
    SELECT *
    FROM read_csv_auto('{DATA_PATH}')
""").df()

summarize_result

In [ ]:
if "year" in schema.names():

    duckdb_year = con.execute(f"""
        SELECT
            year,
            COUNT(*) AS jumlah_data
        FROM read_csv_auto('{DATA_PATH}')
        GROUP BY year
        ORDER BY year
    """).df()

    duckdb_year

## Perbandingan Polars dan DuckDB

Pada tahap profiling ini digunakan dua teknologi:

- **Polars** digunakan untuk membaca dan melakukan transformasi awal
  terhadap dataset dengan pendekatan `LazyFrame` melalui `scan_csv()`.
- **DuckDB** digunakan untuk melakukan analisis berbasis SQL terhadap
  dataset, termasuk perhitungan jumlah baris dan profiling menggunakan
  `SUMMARIZE`.

Pendekatan ini memungkinkan dataset berukuran besar dianalisis tanpa
bergantung pada Pandas sebagai alat utama manipulasi data.

In [ ]:
total_missing = int(
    missing_summary["missing_count"].sum()
)

profiling_summary = pl.DataFrame({
    "Informasi": [
        "Nama Dataset",
        "Jumlah Baris",
        "Jumlah Kolom",
        "Ukuran File (MB)",
        "Jumlah Missing Value",
        "Jumlah Baris Duplikat",
        "Kolom Numerik",
        "Kolom Kategorikal"
    ],
    "Nilai": [
        "Indonesia Palm Oil - Supply Chain Data",
        f"{row_count:,}",
        f"{column_count:,}",
        f"{file_size_mb:,.2f}",
        f"{total_missing:,}",
        f"{duplicate_count:,}",
        len(numeric_columns),
        len(categorical_columns)
    ]
})

profiling_summary

# Kesimpulan

Berdasarkan profiling awal, dataset **Indonesia Palm Oil - Supply Chain
Data** merupakan dataset berukuran besar yang digunakan untuk
menganalisis informasi rantai pasok minyak kelapa sawit Indonesia.

Profiling dilakukan menggunakan **Polars** dan **DuckDB**. Polars
digunakan dengan pendekatan lazy evaluation melalui `scan_csv()` untuk
membaca dan melakukan profiling terhadap dataset, sedangkan DuckDB
digunakan untuk melakukan analisis berbasis SQL.

Beberapa aspek yang diperiksa meliputi ukuran dataset, jumlah baris dan
kolom, struktur data, tipe data, missing values, duplikasi, nilai unik,
statistik deskriptif, serta distribusi data berdasarkan tahun.

Hasil profiling ini menjadi dasar untuk tahap berikutnya. Proses
cleaning, validasi data yang lebih mendalam, penanganan missing value,
duplikasi, dan outlier akan dilakukan pada **Milestone 2** melalui
notebook `02_data_cleaning.ipynb`.

Dataset asli tetap disimpan pada folder `data/raw/` dan tidak diubah.